# 00 · Preparar o ambiente e modelar a base em SQL
[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessicapmartins/previsao-faltas-consultas/blob/main/notebooks/00_setup_modelagem_sql.ipynb)

**O que este notebook faz:**
1. Conecta ao Google Drive.
2. Lê o CSV original (guardado neste repositório, em `data/raw/`).
3. **Modela a base em 3 tabelas** e grava em um banco SQLite (`consultas.db`) na pasta `dados` do projeto:
   - `pacientes`: uma linha por paciente (gênero, bolsa família, condições de saúde)
   - `bairros`: uma linha por bairro
   - `consultas`: uma linha por agendamento (datas, idade, SMS, se faltou)

A base original vem em uma tabela só. Separar em tabelas permite praticar JOINs, como num banco de verdade.
Antes de separar, conferimos que gênero, bolsa família e condições de saúde **não mudam** entre as consultas de um mesmo paciente; por isso podem ficar na tabela `pacientes`.

In [1]:
# Funções de apoio do projeto (mesma carga, mesmas variáveis e mesmo modelo em todos os notebooks)
import os, sys, urllib.request
try:
    import google.colab
    urllib.request.urlretrieve('https://raw.githubusercontent.com/jessicapmartins/previsao-faltas-consultas/main/src/apoio.py', '/content/apoio.py')
    sys.path.insert(0, '/content')
except ModuleNotFoundError:
    sys.path.insert(0, '../src')
import apoio, pandas as pd, numpy as np

con, DADOS, CSV = apoio.conectar()

def q(sql):
    """Roda uma consulta SQL no banco de consultas e devolve um DataFrame."""
    return pd.read_sql(sql, con)

## 1. A carga
O código da carga fica em `src/apoio.py`, para os outros notebooks poderem recriar o banco se o Drive demorar a sincronizar. Ele aparece abaixo para leitura.

In [2]:
import inspect
print(inspect.getsource(apoio.carregar_base))

def carregar_base(con, csv):
    bruto = pd.read_csv(csv)
    df = bruto.rename(columns={
        'PatientId': 'paciente_id', 'AppointmentID': 'consulta_id', 'Gender': 'genero',
        'ScheduledDay': 'data_agendamento', 'AppointmentDay': 'data_consulta', 'Age': 'idade',
        'Neighbourhood': 'bairro', 'Scholarship': 'bolsa_familia', 'Hipertension': 'hipertensao',
        'Diabetes': 'diabetes', 'Alcoholism': 'alcoolismo', 'Handcap': 'deficiencia',
        'SMS_received': 'sms_recebido', 'No-show': 'faltou'})
    df['paciente_id'] = df['paciente_id'].astype('int64')
    df['faltou'] = (df['faltou'] == 'Yes').astype(int)
    df['data_agendamento'] = pd.to_datetime(df['data_agendamento']).dt.tz_localize(None).dt.strftime('%Y-%m-%d %H:%M:%S')
    df['data_consulta'] = pd.to_datetime(df['data_consulta']).dt.strftime('%Y-%m-%d')

    bairros = (df[['bairro']].drop_duplicates().sort_values('bairro')
               .reset_index(drop=True).rename_axis('bairro_id').reset_index())
    pacien

In [3]:
bruto = apoio.carregar_base(con, CSV)
print(bruto.shape)
bruto.head()

(110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


## 2. Os atributos do paciente são fixos?
Se algum paciente tivesse, por exemplo, bolsa família em uma consulta e não em outra, não daria para guardar isso na tabela `pacientes`.

In [4]:
colunas = ['Gender', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap']
(bruto.groupby('PatientId')[colunas].nunique() > 1).sum().rename('pacientes com valor que muda')

Gender          0
Scholarship     0
Hipertension    0
Diabetes        0
Alcoholism      0
Handcap         0
Name: pacientes com valor que muda, dtype: int64

## 3. Conferência
Esperado: 81 bairros, 62.299 pacientes e 110.527 consultas, com cerca de 20% de faltas.

In [5]:
for t in ['bairros', 'pacientes', 'consultas']:
    print(f"{t:<10} {q(f'SELECT COUNT(*) AS n FROM {t}')['n'][0]:>8,} linhas")
q('SELECT ROUND(100.0 * AVG(faltou), 1) AS taxa_falta_pct FROM consultas')

bairros          81 linhas
pacientes    62,299 linhas
consultas   110,527 linhas


,taxa_falta_pct
0,20.2


## Gravar no Google Drive
Fecha o banco e força o Drive a gravar agora. Sem isso, o próximo notebook pode não enxergar as tabelas novas.

In [6]:
con.commit()
con.close()
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Banco gravado no Google Drive.')
except ModuleNotFoundError:
    print('Fora do Colab: nada a sincronizar.')

Fora do Colab: nada a sincronizar.
